# Week 8: Networks I: Roads

The way that we think about connected things like roads, walkways, bus routes, and more differ from the way we think about boundaries. So far, we have primarily dealt with `Polygon` and `Point` spatial objects. `Polygon` objects are things like census tracts, parcel outlines, and buffers around places. `Point` objects are single location things: they often have lat/lon or x/y information that describes where they are, like bikeshare docking locations or housing units. 

Today we are going to work with a different type of spatial object: `LineString`. These are objects that are more than one point (`Point`), but don't encompass any inner space by looping back around to themselves (`Polygon`). `LineString` objects can have elbows or joints that can make them appear approximately curvy. Each linestring segment is made up of as many joints as necessary to match the desired line shape. 

When we string together sets of lines with connecting points between them, we call these **networks**. In network terminolgy, these lines are now called **edges** and the points that connect them together are called **nodes**. You want to have a node everytime you can make a decision between edges. Think about it like a road network: each edge is a road, and it continues until it hits an intersection (node). Even if the road as we know it continues straight, we need a node point to mark that a decision is possible (continue straight, or make a turn). 


## 1 Networkx

`networkx` is a powerful python package for dealing with networks. Like other packages, it encodes some inherent understanding of the special object we care about, in this case networks with edges and nodes. 

In [ ]:
!pip install networkx osmnx

In [ ]:
import networkx as nx 
import numpy as np
import geopandas as gpd
import shapely
import matplotlib.cm as cm 
import matplotlib.pyplot as plt

In [ ]:
# What does a network look like?
G = nx.gnp_random_graph(n=100, p=0.1)

In [ ]:
# We can visualize a graph G
nx.draw(G)

In [ ]:
# We can see how many nodes and edges it has
print(len(G.nodes))
print(len(G.edges))

In [ ]:
# There are many types of networks you can generate, including one that approximates social networks
# and other "small world" connections
H = nx.watts_strogatz_graph(n=100, k=6, p=0.1)

In [ ]:
nx.draw(H)

In [ ]:
# you can assign variables to the nodes (and edges)
# For example, say we want to randomly assign ages
randoms = np.random.randint(low=18, high=90, size=len(H.nodes))
ages = {node: age for node, age in zip(H.nodes, randoms)}
nx.set_node_attributes(H, values=ages, name="age")

In [ ]:
# You can see all the nodes, removing data=False gives the "names" of the nodes
H.nodes(data=True)

In [ ]:
# You can access one node at a time
H.nodes[0]

In [ ]:
# We can also give our edges attributes
# For example, let's randomly assign how far away they live from one another
edge_distances = np.random.randint(low=1, high=100, size=len(H.edges))
distances = {edge: edge_dist for edge, edge_dist in zip(H.edges, edge_distances)}
nx.set_edge_attributes(H, values=distances, name="distance")

In [ ]:
# You can also access the edges
H.edges(data=True)

In [ ]:
# Or the edges of one node
H.edges(0, data=True)

In [ ]:
# We can calculate the shortest path through a network from one node to another
path1 = nx.shortest_path(H, source=0, target=60)
path1

In [ ]:
# We can draw the path, but we need to give the graph plot some "structure"
pos = nx.spring_layout(H)
nx.draw(H, pos, node_color='lightblue', edge_color='gray')

# Define the path edges by building the list of (nodeA, nodeB), (nodeB, nodeC), etc.
path_edges = list(zip(path1, path1[1:]))

# Draw nodes on the path
nx.draw_networkx_nodes(H, pos, nodelist=path1, node_color='red')
# Draw edges on the path
nx.draw_networkx_edges(H, pos, edgelist=path_edges, edge_color='red', width=2)

In [ ]:
## YOUR TURN 
## Add another attribute to the edges (maybe how often the nodes hang out together?)
## Compute the shortest path using THAT attribute
## Plot it on the graph with our red path



## 2 OSMnx

OSMnx lets you download, model, analyze, and visualize street networks (and any other spatial data) anywhere in the world from OpenStreetMap.

OSMnx is built on top of GeoPandas, NetworkX, and matplotlib and interacts with OpenStreetMap’s APIs to:
* Download and model street networks or other networked infrastructure anywhere in the world with a single line of code
* Download any other spatial geometries, place boundaries, building footprints, or points of interest as a GeoDataFrame
* Download by city name, polygon, bounding box, or point/address + network distance
* Download drivable, walkable, bikeable, or all street networks
* Download node elevations and calculate edge grades (inclines)
* Impute missing speeds and calculate graph edge travel times
* Simplify and correct the network’s topology to clean-up nodes and consolidate intersections
* Fast map-matching of points, routes, or trajectories to nearest graph edges or nodes
* Save networks to disk as shapefiles, GeoPackages, and GraphML
* Save/load street network to/from a local .osm XML file
* Conduct topological and spatial analyses to automatically calculate dozens of indicators
* Calculate and visualize street bearings and orientations
* Calculate and visualize shortest-path routes that minimize distance, travel time, elevation, etc
* Visualize street networks as a static map or interactive Leaflet web map
* Visualize travel distance and travel time with isoline and isochrone maps
* Plot figure-ground diagrams of street networks and building footprints
More info:

[OSMnx documentation](https://osmnx.readthedocs.io/en/stable/)

[Examples, demos, tutorials](https://github.com/gboeing/osmnx-examples)



In [ ]:
import osmnx as ox

In [ ]:
# Now we can start from a place name (this can take a little bit of time depending on the size)
place = "Piedmont, California, USA"
G = ox.graph.graph_from_place(place, network_type="drive")
# network_type options = all, all_public, bike, drive, drive_service, walk
fig, ax = ox.plot.plot_graph(G, bgcolor='white', node_color='black', edge_color='black')

In [ ]:
# We can still look at the nodes and edges
tmp = list(G.nodes)[0:10] # These are OSM ids
tmp

In [ ]:
# And they have some data
G.nodes(data=True)[53017091]

In [ ]:
# Same with our edges
list(G.edges)[0:10]

In [ ]:
# The edges have data too
G[53017091][53064327]

In [ ]:
# If it's helpful, you can convert the graph to a GeoDataFrame
gdf_nodes, gdf_edges = ox.convert.graph_to_gdfs(G)
gdf_edges.head()

In [ ]:
# what sized area does our network cover in square meters?
G_proj = ox.projection.project_graph(G) # This changes the underlying crs of the network to local meters
nodes_proj = ox.convert.graph_to_gdfs(G_proj, edges=False)

In [ ]:
ox.projection.project_graph?

In [ ]:
graph_area_m = nodes_proj.union_all().convex_hull.area
graph_area_m

In [ ]:
# You can also see some basic stats about the graph
ox.stats.basic_stats?

In [ ]:
ox.stats.basic_stats(G_proj, area=graph_area_m, clean_int_tol=15)

In [ ]:
# We can still set variables to the nodes and edges
# For example, let's set a random variable to each edge representing "beauty" on a scale of 1 to 100
rand_beauty = np.random.randint(low=1, high=100, size=len(G.edges))
beauty_values = {edge: beauty for edge, beauty in zip(G.edges, rand_beauty)}
nx.set_edge_attributes(G, beauty_values, "edge_beauty")
G[53017091][53064327] # Our example edge

In [ ]:
# color edges in original graph with beauty values
ec = ox.plot.get_edge_colors_by_attr(G, "edge_beauty", cmap="inferno")
fig, ax = ox.plot.plot_graph(G, edge_color=ec, bgcolor='white', edge_linewidth=2, node_size=0)

In [ ]:
## YOUR TURN 
## Try to download another network, set a value to nodes, and plot it
## Caution: try to stick to smaller networks for now, they can take a little bit to download


## 3 Routing

As with the shortest path calculations, we can do routing via the attributes of our edges in the road network. 

In [ ]:
# OSM tries to gather the speed limits of most roads
G = ox.routing.add_edge_speeds(G) # in km per hour
# Then it calculates travel time using the length of the segment and the speed of the road
G = ox.routing.add_edge_travel_times(G) # in seconds 

In [ ]:
# Let's look at our example edge again
G[53017091][53064327] # Our example edge

In [ ]:
# We can then plot our edges by these speeds
ec = ox.plot.get_edge_colors_by_attr(G, "speed_kph", cmap="inferno")
fig, ax = ox.plot.plot_graph(G, edge_color=ec, bgcolor='white', edge_linewidth=2, node_size=0)

In [ ]:
# get the nearest network nodes to two lat/lng points with the distance module
orig = ox.distance.nearest_nodes(G, X=-122.245846, Y=37.828903)
dest = ox.distance.nearest_nodes(G, X=-122.215006, Y=37.812303)

In [ ]:
# find the shortest path between nodes, minimizing travel time, then plot it
route = ox.routing.shortest_path(G, orig, dest, weight="travel_time")
fig, ax = ox.plot.plot_graph_route(G, route, bgcolor='white', node_size=0)

In [ ]:
## YOUR TURN 
## Find the shortest path between the nodes based on travel distance instead of time


## 4 Other things you can download in OSMnx

Buildings! Amenities! Parks!

In [ ]:
# get all building footprints in some neighborhood
place = "Chinatown, San Francisco, California"
tags = {"building": True}
gdf = ox.features.features_from_place(place, tags)

In [ ]:
gdf.head()

In [ ]:
gdf.plot()

In [ ]:
# get all parks and bus stops in some neighborhood
tags = {"leisure": "park", "highway": "bus_stop"}
gdf = ox.features.features_from_place(place, tags)
gdf.iloc[0]

In [ ]:
gdf.plot()

In [ ]:
## YOUR TURN 
## Explore the OSM tags and download a new feature in a new place


## 5 Other ways to get networks

In [ ]:
# You can start with a point and a buffer (as the crow flies)

# define a point at the corner of California St and Mason St in SF
location_point = (37.791427, -122.410018)

# create network from point, inside bounding box of N, S, E, W each 750m from point
G = ox.graph.graph_from_point(location_point, dist=750, dist_type="bbox", network_type="drive")
fig, ax = ox.plot.plot_graph(G, node_color="r", bgcolor='white')

In [ ]:
# Or start with a point and a buffer (distance via the network)

# same point again, but create network only of nodes within 500m along the network from point
G = ox.graph.graph_from_point(location_point, dist=500, dist_type="network")
fig, ax = ox.plot.plot_graph(G, node_color="r", bgcolor='white')

In [ ]:
# Or a bounding box

# define a bounding box in San Francisco as (left, bottom, right, top)
bbox = -122.43, 37.78, -122.41, 37.79

# create network from that bounding box
G = ox.graph.graph_from_bbox(bbox, network_type="drive_service")
fig, ax = ox.plot.plot_graph(G, bgcolor='white', edge_linewidth=2, node_size=0)

In [ ]:
# Or an address and a distance

# network from address, including only nodes within 1km along the network from the address
G = ox.graph.graph_from_address(
    address="350 5th Ave, New York, NY",
    dist=1000,
    dist_type="network",
    network_type="drive",
)

# you can project the network to UTM (zone calculated automatically)
G_projected = ox.projection.project_graph(G)
fig, ax = ox.plot.plot_graph(G_projected, bgcolor='white', edge_linewidth=2, node_size=0)

In [ ]:
# Or a polygon from a shapefile?

nyc = gpd.read_file("nyct2020.zip")
nyc = nyc.to_crs(epsg=4326)
les = nyc[nyc["NTAName"] == "Lower East Side"].union_all() # only in lower east side neighborhood
G2 = ox.graph.graph_from_polygon(les, network_type="drive")
fig, ax = ox.plot.plot_graph(G2, bgcolor='white', edge_linewidth=2, node_size=0)